## Enzyme types

In [ ]:
from cobra.io import load_model
from GEM_dict import count_enzyme_types, count_enzyme_types2
 
model_ecoli = load_model("iML1515")    
dictionary_ecoli, counts_ecoli = count_enzyme_types(model_ecoli)
df_ecoli, counts_ecoli2 = count_enzyme_types2(model_ecoli)
print(counts_ecoli) 
print('total enzymes: ', sum(counts_ecoli.values()))

In [ ]:
from cobra.io import read_sbml_model

modelhuman = read_sbml_model(r"C:\Users\lyach\OneDrive - University of Toronto\GitHub\CatNivoDataset\data\GEMs\Human-GEM.xml")
dictionary_human, counts_human = count_enzyme_types(modelhuman)
print(counts_human) 
print('total enzymes: ', sum(counts_human.values()))

In [ ]:
from cobra.io import read_sbml_model

modelhuman = read_sbml_model(r"C:\Users\lyach\OneDrive - University of Toronto\GitHub\CatNivoDataset\data\GEMs\Human-GEM.xml")
dictionary_human, counts_human = count_enzyme_types(modelhuman)
print(counts_human) 
print('total enzymes: ', sum(counts_human.values()))

In [ ]:
from cobra.io import read_sbml_model

modelhuman = read_sbml_model(r"C:\Users\lyach\OneDrive - University of Toronto\GitHub\CatNivoDataset\data\GEMs\Human-GEM.xml")
dictionary_human, counts_human = count_enzyme_types(modelhuman)
print(counts_human) 
print('total enzymes: ', sum(counts_human.values()))

In [ ]:
from cobra.io import read_sbml_model
from GEM_dict import count_enzyme_types, count_enzyme_types2


modelyeast = read_sbml_model(r"C:\Users\lyach\OneDrive - University of Toronto\GitHub\CatNivoDataset\data\GEMs\yeast-GEM.xml")
dictionary_yeast, counts_yeast = count_enzyme_types(modelyeast)
print(counts_yeast) 
print('total enzymes: ', sum(counts_yeast.values()))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Data for each organism
organisms_data = {
    'Human GEM (H. sapiens)': {
        'total': 8046,
        'Homomeric': 3478,
        'Isoenzymes': 3988,
        'Complexes': 577
    },
    'iML1515 (E. coli)': {
        'total': 2266,
        'Homomeric': 1302,
        'Isoenzymes': 743,
        'Complexes': 221
    },
    'Yeast9 (S. cerevisiae)': {
        'total': 2709,
        'Homomeric': 1959,
        'Isoenzymes': 588,
        'Complexes': 159
    }
}

# Colors for enzyme types
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']  # Blue, Orange, Green
enzyme_types = ['Homomeric', 'Isoenzymes', 'Complexes']

# Create figure with subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Enzyme Types in GEMs', fontsize=16, y=1.06)

# Create pie charts for each organism
for idx, (organism, ax) in enumerate(zip(organisms_data.keys(), axes)):
    # Get counts for this organism
    counts = [organisms_data[organism][enzyme_type] for enzyme_type in enzyme_types]
    
    # Calculate percentages for labels
    total = organisms_data[organism]['total']
    
    # Create labels with both count and percentage
    labels = [f'{enzyme_type}\n{count:,}' 
              for enzyme_type, count in zip(enzyme_types, counts)]
    
    # Create pie chart
    wedges, texts, autotexts = ax.pie(counts, labels=labels, colors=colors, 
                                     autopct='', startangle=90, 
                                     wedgeprops=dict(width=0.8))
    
    # Customize the appearance
    ax.set_title(f'{organism}\nTotal: {organisms_data[organism]["total"]:,} enzymes', 
                fontsize=18, pad=20)
    
    # Adjust text properties
    for text in texts:
        text.set_fontsize(14)

plt.show()

# Print summary statistics
print("Enzyme Type Distribution Summary:")
print("=" * 60)
for organism in organisms_data.keys():
    print(f"\n{organism} (Total: {organisms_data[organism]['total']:,} enzymes):")
    print("-" * 40)
    for enzyme_type in enzyme_types:
        count = organisms_data[organism][enzyme_type]
        print(f"  {enzyme_type:12s}: {count:5,}")

## Homomeric enzymes using Sympy parser

In [14]:
from cobra.io import load_model
from GEM_dict import count_enzyme_types, count_enzyme_types2
import sympy
from sympy.logic.boolalg import to_dnf
from sympy import sympify, Symbol, Or

model = load_model("iML1515")

enzyme_dict = {
    "homomeric": [],    # list of (reaction_id, gene_id) tuples
    "isoenzymes": [],   # list of ([reaction_ids], gene_id) tuples  
    "complexes": []     # list of ([reaction_ids], [gene_ids]) tuples
}
gene_hits = {}        # gene-id ➜ number of reactions it appears in
homomeric_genes = set()

for rxn in model.reactions:
    gpr = rxn.gene_reaction_rule.strip()
    if not gpr:                 # 1
        continue
    expr = gpr.lower().replace(" and "," & ").replace(" or "," | ")
    dnf  = to_dnf(sympify(expr), simplify=True, force=True)
    
    if isinstance(dnf, Symbol):
        # Homomeric enzyme: single gene catalyzes this reaction
        gene_id = str(dnf)
        enzyme_dict["homomeric"].append((rxn.id, gene_id, gpr))
        homomeric_genes.add(gene_id)
        
    elif isinstance(dnf, Or) and all(isinstance(a, Symbol) for a in dnf.args):
        # Isoenzymes: multiple genes can independently catalyze this reaction
        for gene_symbol in dnf.args:
            gene_id = str(gene_symbol)
            # Find existing entry for this gene or create new one
            existing_entry = None
            for i, (reaction_ids, existing_gene_id) in enumerate(enzyme_dict["isoenzymes"]):
                if existing_gene_id == gene_id:
                    existing_entry = i
                    break
            
            if existing_entry is not None:
                # Add this reaction to existing gene's reaction list
                enzyme_dict["isoenzymes"][existing_entry] = (
                    enzyme_dict["isoenzymes"][existing_entry][0] + [rxn.id],
                    gene_id
                )
            else:
                # Create new entry for this gene
                enzyme_dict["isoenzymes"].append(([rxn.id], gene_id))
                
    else:
        # Complex: multiple genes required together (AND logic)
        gene_ids = [g.id for g in rxn.genes]
        # Find existing entry with same gene combination or create new one
        existing_entry = None
        for i, (reaction_ids, existing_gene_ids) in enumerate(enzyme_dict["complexes"]):
            if set(existing_gene_ids) == set(gene_ids):
                existing_entry = i
                break
        
        if existing_entry is not None:
            # Add this reaction to existing complex's reaction list
            enzyme_dict["complexes"][existing_entry] = (
                enzyme_dict["complexes"][existing_entry][0] + [rxn.id],
                enzyme_dict["complexes"][existing_entry][1]
            )
        else:
            # Create new entry for this gene combination
            enzyme_dict["complexes"].append(([rxn.id], gene_ids))
    
    for g in rxn.genes:                     # track global usage
        gene_hits[g.id] = gene_hits.get(g.id, 0) + 1


In [ ]:
iso1 = model.reactions.get_by_id("HXPRT")
print(iso1.gene_reaction_rule)

complex1 = model.reactions.get_by_id("GLYCTO2")
print(complex1.gene_reaction_rule)

complex2 = model.reactions.get_by_id("GLYCTO3")
print(complex2.gene_reaction_rule)

complex3 = model.reactions.get_by_id("GLYCTO4")
print(complex3.gene_reaction_rule)

# Checking exchanges (**discarded**)

Discarded

In [ ]:
# Df with NO BiGG matches -- just exchange reactions
eco_out_misses_exchanges = eco_out_misses[
    eco_out_misses['ecomics_rxn'].str.contains('exchange|Exch|ext', case=False, na=False)
]

# Exporting for manual curation
eco_out_misses_exchanges = eco_out_misses_exchanges.iloc[:, [0, 1]]
eco_out_misses_exchanges.to_csv(r"C:\Users\lyach\OneDrive - University of Toronto\GitHub\CatNivoDataset\data\LiteratureDatasets\eco_out_misses_exchanges.csv", index=False)

In [1]:
import cobra
from cobra.io import load_model

model = load_model("iML1515")

In [ ]:
for reaction in model.reactions:
    print(reaction)

In [20]:
# Get exchange reactions
exchange_rxns = [rxn for rxn in model.exchanges]

# Create DataFrame
df_exchanges = pd.DataFrame({
    "gem_rxn_id": [rxn.id for rxn in exchange_rxns],
    "gem_rxn":    [rxn.reaction for rxn in exchange_rxns]
})

# Exporting for manual curation
df_exchanges.to_csv(r"C:\Users\lyach\OneDrive - University of Toronto\GitHub\CatNivoDataset\data\LiteratureDatasets\iml1515_exchanges.csv", index=False)

In [ ]:
for rxn in exchange_rxns:
    print()
    print(rxn.id, ',', rxn.reaction)

# Matching ECOMICS to iML1515 rxn IDs

Initial match using Dice coefficient to measure similarity between metabolites -- discarded (June)

In [16]:
import re, pandas as pd
from pathlib import Path

def parse_metabolites(rxn: str) -> set[str]:
    """
    Convert a reaction string → set of metabolite ‘tokens’.
    Removes stoichiometric coefficients, compartment tags (_c, _e, …),
    parentheses and arrow direction.
    """
    if pd.isna(rxn) or not isinstance(rxn, str):
        return set()

    s = rxn.lower().replace('<=>', '->').replace('-->', '->')

    if '->' not in s:
        return set()  # skip invalid reactions

    try:
        left, right = s.split('->', 1)
    except ValueError:
        return set()

    parts = re.split(r'\s*\+\s*', left) + re.split(r'\s*\+\s*', right)

    mets = []
    for p in parts:
        p = re.sub(r'^\d+(\.\d+)?\s*', '', p)         # drop coefficient
        p = re.sub(r'_[a-z]\w*$', '', p)              # drop compartment
        p = p.strip('() ').strip()
        if p:
            mets.append(p)
    return set(mets)

def dice(a: set, b: set) -> float:
    # Dice coefficient to measure similarity between metabolites
    return 0 if not a or not b else 2 * len(a & b) / (len(a) + len(b))



In [17]:
eco = pd.read_csv(r"C:\Users\lyach\OneDrive - University of Toronto\GitHub\CatNivoDataset\data\LiteratureDatasets\ECOMICS\ecomics_rxns.csv")
iml = pd.read_csv(r"C:\Users\lyach\OneDrive - University of Toronto\GitHub\CatNivoDataset\data\LiteratureDatasets\iml1515_rxns.csv")

# pre-compute token sets once
iml["token_set"] = iml["gem_rxn"].apply(parse_metabolites)

# match rxns between ecomics and gem
matches = []
for _, row in eco.iterrows():
    # 1. exact BiGG hit if available
    hit = iml.loc[iml["gem_bigg"] == row["bigg_id"]].head(1)

   # 2. otherwise best Dice score using tokenised reactions
    score = pd.NA
    if hit.empty:
        tgt_tokens = parse_metabolites(row["ecomics_rxn"])
        scores = iml["token_set"].apply(lambda x: dice(tgt_tokens, x))
        best   = scores.idxmax()
        score  = scores.at[best]
        hit    = iml.loc[[best]] if score > 0 else pd.DataFrame()

    # 3. no good match found
    if hit.empty:
        matches.append({
            "gem_rxn_match": pd.NA,
            "dice_score": pd.NA,
            "gem_rxn_id": pd.NA
        })
    else:
        h = hit.iloc[0]
        matches.append({
            "gem_rxn_match": h["gem_rxn"],
            "dice_score": round(score, 3) if score is not pd.NA else pd.NA,
            "gem_rxn_id": h["gem_rxn_id"]
        })

eco_out = eco.join(pd.DataFrame(matches))

In [5]:
# look at least confident Dice scores
eco_out_clean = eco_out.dropna(subset=['dice_score']).sort_values(by='dice_score', ascending=True)
eco_out_clean = eco_out_clean.sort_values(by='dice_score', ascending=True)
eco_out_clean['to check'] = None
eco_out_clean['ecomics_rxn_type'] = None
#eco_out_clean.to_csv(r"C:\Users\lyach\OneDrive - University of Toronto\GitHub\CatNivoDataset\data\LiteratureDatasets\eco_clean.csv", index=False)

In [6]:
# the ones wo Dice score
eco_out_miss = eco_out.sort_values(by='gem_rxn_id', ascending=True)
eco_out_miss = eco_out_miss.tail(10)
#eco_out_miss.to_csv(r"C:\Users\lyach\OneDrive - University of Toronto\GitHub\CatNivoDataset\data\LiteratureDatasets\eco_miss.csv", index=False)

In [ ]:
# Confident reactions to work with!
# These were checked with an LLM by comparing the GEM rxn and the ECOMICS rxn
eco_checked = pd.read_csv(r"C:\Users\lyach\OneDrive - University of Toronto\GitHub\CatNivoDataset\data\LiteratureDatasets\eco_checked.csv")

# Store the names of these rxns we're confident the matching makes sense (to check = False)
eco_conf_list = eco_checked.loc[eco_checked['to_check'] == False, 'ecomics_rxn_id'].tolist()

In [ ]:
# make a confident dataframe with the conf list
# and also keeping the matching bigg ids
eco_conf = eco_out[
    (eco_out['ecomics_rxn_id'].isin(eco_conf_list)) |
    (eco_out['bigg_id'] == eco_out['gem_rxn_id'])
]

In [ ]:
# keep just useful columns
eco_conf = eco_conf.iloc[:, [0, -1]]

In [ ]:
ecomicsfluxes = pd.read_csv(r"C:\Users\lyach\OneDrive - University of Toronto\GitHub\CatNivoDataset\data\LiteratureDatasets\ECOMICS\fluxomics_ecomics.csv")